# Exercise 17 - Simple Transformer, Open-weight LLM and LoRA

**There are 3 sections in this exercise**

Part A: Train a Simple Tranformer Encoder Model

Part B: Load and execute a open-weight LLM model

Part C: Fine Tune model using LoRA (Similar to Transfer Learning)

- Recommended Hardware accelerator: **T4 GPU**

Reference code and APIs from the following sources:

1. Transformer Experiments by [David Cardozo](https://github.com/Davidnet/transformer-experiments)
2. Keras Hub [modeling layers](https://keras.io/keras_hub/api/modeling_layers/)
3. Google AI for developers - [Run Gemma with Keras](https://ai.google.dev/gemma/docs/core/keras_inference)
4. Keras example - [Parameter-efficient fine-tuning of Gemma with LoRA and QLoRA](https://keras.io/examples/keras_recipes/parameter_efficient_finetuning_of_gemma_with_lora_and_qlora/) by Hongyu Chiu, Abheesht Sharma, Matthew Watson

--------------------------------------------------------------------------------

### **Part A: Train a Simple Transformer Model - Use GPU (Training Task)**

Building a transformer from scratch in Keras involves creating several custom layers, including multi-head attention and positional encoding, and then stacking them to form an encoder and/or decoder. The following is a simplified, working example of a transformer encoder model for text classification.

### The Transformer Block

The core of a transformer model is the **Transformer Block**, which combines a multi-head attention layer and a feed-forward network. Both are wrapped with dropout, residual connections, and layer normalization. This is implemented as a custom Keras `Layer`.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training):
        # Multi-head attention layer
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        # Add and Norm
        out1 = self.layernorm1(inputs + attn_output)

        # Feed-forward network
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        # Add and Norm
        return self.layernorm2(out1 + ffn_output)

### Positional Embedding

Transformers lack the recurrent or convolutional components that give other models a sense of word order. To overcome this, we add **positional encodings** to the word embeddings. This provides the model with information about the relative position of each token in the sequence.

In [ ]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

###  Build the Model

Now, let's create a full transformer model for a classification task (e.g., movie review sentiment analysis). This model will use the layers we just defined.

In [ ]:
vocab_size = 20000  # Only consider the top 20k words
maxlen = 200  # Only consider the first 200 words of each movie review
embed_dim = 32  # Embedding size for each token
num_heads = 2  # Number of attention heads
ff_dim = 32  # Hidden layer size in feed forward network

inputs = layers.Input(shape=(maxlen,))
embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
x = embedding_layer(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x, training=True)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(20, activation="relu")(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(2, activation="softmax")(x)

model = keras.Model(inputs=inputs, outputs=outputs)
model.summary()

### Train the Model and then test on validation data

You can now compile and train the model using a standard Keras workflow. Here's an example with the IMDB movie review dataset.

In [ ]:
import keras

(x_train, y_train), (x_val, y_val) = keras.datasets.imdb.load_data(num_words=vocab_size)
print(f"Training data shape: {x_train.shape}")
print(f"Validation data shape: {x_val.shape}")

# Pad sequences to a fixed length
x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen)
x_val = keras.preprocessing.sequence.pad_sequences(x_val, maxlen=maxlen)

# Compile and train
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
history = model.fit(
    x_train, y_train, batch_size=32, epochs=5, validation_data=(x_val, y_val)
)

print("\nTesting the model on the validation dataset...")
loss, accuracy = model.evaluate(x_val, y_val, verbose=1)

print(f"\nTest loss: {loss:.4f}")
print(f"Test accuracy: {accuracy:.4f}")

### **Part B:  Load and Execute Open-Weight Gemma3 model with Keras - Use either CPU or GPU (Inference Task)**



### Setup

Visit this website, https://www.kaggle.com/models/google/gemma-3 and accept the terms for use of Gemma3 models by clicking on Terms in the line,
Terms of Use: Terms.  Once successfully completed, on the webpage, you should see a line - *You've consented to the license for Gemma 3*

### Install Keras packages

Install the Keras and KerasHub Python packages.

In [ ]:
!pip install -q -U keras-hub
!pip install -q -U keras

### Select a backend

Keras is a high-level, multi-framework deep learning API designed for simplicity and ease of use. For this tutorial, configure the backend for JAX as it typically provides the better performance.

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"  # Or "tensorflow" or "torch".
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

### Import packages

Import the Keras and KerasHub packages.

In [ ]:
import keras
import keras_hub

## Load model

Keras provides implementations of many popular [model architectures](https://keras.io/api/keras_nlp/models/). Download and configure a Gemma model using the `Gemma3CausalLM` class to build an end-to-end, causal language modeling implementation for Gemma 3 models. Create the model using the `from_preset()` method, as shown in the following code example:

This is an instruction-tuned (IT)version of Gemma modes that is  trained to follow instructions and perform tasks such as summarization, question answering, and coding.

In [ ]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset(
    "gemma3_instruct_270m",
    dtype="bfloat16",
)

In [ ]:
gemma_lm.summary()

The output of the summary shows the models total number of trainable parameters.
For purposes of naming the model, the embedding layer is not counted against the number of parameters.

### Question on Model Summary::

- If the datatype (dtype="bfloat16") is set to default, what would be the size of the model.  Presently it is 511.36 MB
- What is the purpose of detokenizer (the last layer of the model)

## Generate text with text

Generate text with a text prompt with using `generate()` method of the Gemma model object you configured in the previous steps.
- The optional `max_length` argument specifies the maximum length of the generated sequence.
- Gemma3 output is formatted as Markdown

The following code examples shows a few ways to prompt the model.

In [ ]:
from IPython.display import display, Markdown

answer_text = gemma_lm.generate("Open-weights models are useful because", max_length=400)

# Join the generated text into a single string and display as Markdown
display(Markdown("".join(answer_text)))

You can also provide batched prompts using a list as input:

In [ ]:
from IPython.display import display, Markdown

response_text = gemma_lm.generate(
    ["Gemma3 architecture differs from Llama3",
     "LLMs are useful because"], )

# Display each response using Markdown formatting
for response in response_text:
  display(Markdown(response))

If you're running on JAX or TensorFlow backends, you should notice that the second `generate()` call returns an answer more quickly. This performance improvement is because each call to `generate()` for a given batch size and `max_length` is compiled with XLA. The first run is expensive, but subsequent runs are faster.

### Use a prompt template

When building more complex requests or multi-turn chat interactions use a prompt template to structure your request. The following code creates a standard template for Gemma prompts:

In [ ]:
PROMPT_TEMPLATE = """<start_of_turn>user
{question}
<end_of_turn>
<start_of_turn>model
"""

The following code shows how to use the template to format a simple request:

In [ ]:
question = """"what are LLMs in 3 bullet points?"""
prompt = PROMPT_TEMPLATE.format(question=question)
output_text = gemma_lm.generate(prompt)

# Display the response using Markdown formatting
display(Markdown(output_text))

### **Part C :  Fine Tune Model using LoRA - Use GPU (Transfer Learning/Training Task)**

[Low Rank Adaptation](https://arxiv.org/abs/2106.09685) (LoRA) is a fine-tuning technique which greatly reduces the number of trainable parameters for downstream tasks by freezing the weights of the model and inserting a smaller number of new weights into the model. This technique makes training with LoRA much faster and more memory-efficient, and produces smaller model weights (a few hundred MBs), all while maintaining the quality of the model outputs. This tutorial walks you through using Keras to perform LoRA fine-tuning on a Gemma model.

### Import packages *(Step not needed if continuing from Part B)*



In [ ]:
import keras
import keras_hub

## Load model *(Step not needed if continuing from Part B)*

- Keras provides implementations of Gemma and many other popular [model architectures](https://keras.io/keras_hub/api/models/). Use the `Gemma3CausalLM.from_preset()` method to configure an end-to-end Gemma model for causal language modeling. A causal language model predicts the next token based on previous tokens.
- Model fine-tuning is possible because the weights of the model are available

In [ ]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_instruct_270m")
gemma_lm.summary()

## Inference before fine tuning

Once you have downloaded and configured a Gemma model, you can query it with various prompts to see how it responds.

### Design considerations prompt

Query the model for suggestions on how to design an AI application

In [ ]:
from IPython.display import display, Markdown

template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

prompt = template.format(
    instruction="How to design an AI application",
    response="",
)
sampler = keras_hub.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler=sampler)

reply_text = gemma_lm.generate(prompt, max_length=512)

# Display the response using Markdown formatting
display(Markdown(reply_text))

The model responds with generic tips on how to create an AI application

### Photosynthesis prompt

Prompt the model to explain photosynthesis in terms simple enough for a 5 year old child to understand.

In [ ]:
from IPython.display import display, Markdown

prompt = template.format(
    instruction="Explain the process of photosynthesis in a way that a child could understand.",
    response="",
)

text_output = gemma_lm.generate(prompt, max_length=256)

display(Markdown(text_output))

The model response contains words that might not be easy to understand for a child such as chlorophyll.

## LoRA fine-tuning

This section shows you how to do fine-tuning using the Low Rank Adaptation (LoRA) tuning technique. This approach allows you to change the behavior of Gemma models using fewer compute resources.

### Load dataset

Prepare a dataset for tuning by downloading an existing data set and formatting if for use with the the Keras `fit()` fine-tuning method. This tutorial uses the [Databricks Dolly 15k dataset](https://huggingface.co/datasets/databricks/databricks-dolly-15k) for fine-tuning. The dataset contains 15,000 high-quality human-generated prompt and response pairs specifically designed for tuning generative models.

In [ ]:
!wget -O databricks-dolly-15k.jsonl https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl

View the JSON Line file to understand how the 15,000 prompt and response pairs look like.

In [ ]:
!cat databricks-dolly-15k.jsonl | jq . | head -n 25

### Format tuning data

Format the downloaded data for use with the Keras `fit()` method. The following code extracts a subset of the training examples to execute the notebook faster. Consider using more training data for higher quality fine-tuning.

In [ ]:
import json

prompts = []
responses = []
line_count = 0

with open("databricks-dolly-15k.jsonl") as file:
    for line in file:
        if line_count >= 1000:
            break  # Limit the training examples, to reduce execution time.

        examples = json.loads(line)
        # Filter out examples with context, to keep it simple.
        if examples["context"]:
            continue
        # Format data into prompts and response lists.
        prompts.append(examples["instruction"])
        responses.append(examples["response"])

        line_count += 1

data = {
    "prompts": prompts,
    "responses": responses
}

### Configure LoRA tuning

Activate LoRA tuning using the Keras `model.backbone.enable_lora()` method, including a LoRA rank value. The *LoRA rank* determines the dimensionality of the trainable matrices that are added to the original weights of the LLM. It controls the expressiveness and precision of the fine-tuning adjustments. A higher rank means more detailed changes are possible, but also means more trainable parameters. A lower rank means less computational overhead, but potentially less precise adaptation.

This example uses a LoRA rank of 4. In practice, begin with a relatively small rank (such as 4, 8, 16). This setting is computationally efficient for experimentation. Train your model with this rank and evaluate the performance improvement on your task. Gradually increase the rank in subsequent trials and see if that further boosts performance.

In [ ]:
# Enable LoRA for the model and set the LoRA rank to 4.
gemma_lm.backbone.enable_lora(rank=2)

Check the model summary after setting the LoRA rank. Notice that enabling LoRA reduces the number of trainable parameters significantly compared to the total number of parameters in the model:

In [ ]:
gemma_lm.summary()

Configure the rest of the fine-tuning settings, including the preprocessor settings, optimizer, number of tuning epochs, and batch size:

In [ ]:
# Limit the input sequence length to 256 (to control memory usage).
gemma_lm.preprocessor.sequence_length = 256
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
# Exclude layernorm and bias terms from decay.
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)

### Run the fine-tune process

Run the fine-tuning process using the `fit()` method. This process can take several minutes depending on your compute resources, data size, and number of epochs:

#### Mixed precision fine-tuning on NVIDIA GPUs

Full precision is recommended for fine-tuning. When fine-tuning on NVIDIA GPUs, you can use mixed precision (`keras.mixed_precision.set_global_policy('mixed_bfloat16')`) to speed up training with minimal effect on training quality.

In [ ]:
# Enable mixed precision training on GPUs
keras.mixed_precision.set_global_policy('mixed_bfloat16')

In [ ]:
gemma_lm.fit(data, epochs=1, batch_size=1)

## Inference after fine-tuning

After fine-tuning, you should see changes in the responses when the tuned model is given the same prompt.

### Photosynthesis prompt

Try the photosynthesis explanation prompt from earlier and note the differences in the response.

In [ ]:
prompt = template.format(
    instruction="Explain the process of photosynthesis in a way that a child could understand.",
    response="",
)

tuned_text = gemma_lm.generate(prompt, max_length=256)

display(Markdown(tuned_text))

The model now explains photosynthesis in simpler terms.

## Improving fine-tune results

For demonstration purposes, this tutorial fine-tunes the model on a small subset of the dataset for just one epoch and with a low LoRA rank value. To get better responses from the fine-tuned model, you can experiment with:

1. Increasing the size of the fine-tuning dataset
2. Training for more steps (epochs)
3. Setting a higher LoRA rank
4. Modifying the hyperparameter values such as `learning_rate` and `weight_decay`.